# P4 Agent 4 · Gold Evaluation Structure

| 항목 | 명세 |
|---|---|
| 목적 | Execute the zero-row gold contract and close as NOT_EVALUATED without performance claims. |
| 담당 Agent | `P4-A4-NCS` |
| Stage ID | `A4-05-EVALUATE` |
| 입력 | `ncs_mapping/data/gold/ncsMappings/gold_ncs_mapping_v1_TEMPLATE.csv` |
| 처리 | zero-row gold 구조 실행 및 NOT_EVALUATED 종료 |
| 출력 | null precision/finalCoverage evaluation artifact 및 4개 종료 artifact |
| 선행 Gate | `NCS_GOLD_SAMPLE_READY` |
| 후속 활용 | future production gold evaluation only |

> Development-only orchestration. Empirical analysis and production promotion are disabled.

In [1]:
RUN_MODE = "observed-dev"
AGENT_ID = "P4-A4-NCS"
STAGE_ID = "A4-05-EVALUATE"
CONTRACT_VERSION = "2.1.2"
SCHEMA_VERSION = "ncs-evaluation-v1"
DATA_VERSION = "observed-dev-20260806.1"
CRAWL_RELEASE_ID = "CRAWL_20260806_03"
AS_OF_DATE = "2026-08-06"
INPUT_MANIFEST_PATH = "ncs_mapping/data/gold/ncsMappings/gold_ncs_mapping_v1_TEMPLATE.csv"
OUTPUT_ROOT = "ncs_mapping/data/runs/observed-dev/NCS_MAPPING_OBSERVED_20260806_01/A4-05-EVALUATE"
RANDOM_SEED = 20260806
FAIL_ON_GATE = True
EMPIRICAL_ANALYSIS_ALLOWED = False
DATA_PROVENANCE = "OBSERVED_DEVELOPMENT_ONLY"
PROMOTION_ALLOWED = False
DUTY_INPUT_PATH = ""
GOLD_INPUT_PATH = ""
CONTROL_SCHEMA_DIR = ""

In [2]:
from pathlib import Path
import os
import sys
import pandas as pd

NCS_ROOT = Path.cwd().resolve()
if NCS_ROOT.name != 'ncs_mapping':
    raise RuntimeError('run this notebook with cwd=ncs_mapping')
sys.path.insert(0, str(NCS_ROOT / 'src'))
assert RUN_MODE == 'observed-dev'
assert AGENT_ID == 'P4-A4-NCS' and STAGE_ID.startswith('A4-')
assert RANDOM_SEED == 20260806 and FAIL_ON_GATE is True
assert DATA_PROVENANCE == 'OBSERVED_DEVELOPMENT_ONLY'
assert EMPIRICAL_ANALYSIS_ALLOWED is False and PROMOTION_ALLOWED is False
resolved_duty_input = DUTY_INPUT_PATH or os.environ.get('P4_A2_DUTY_HANDOFF', '')
resolved_gold_input = GOLD_INPUT_PATH or os.environ.get('P4_NCS_GOLD_INPUT', '')
resolved_schema_dir = CONTROL_SCHEMA_DIR or os.environ.get('P4_CONTROL_SCHEMA_DIR', '')

In [3]:
from p4_ncs.evaluation.gold_evaluation import evaluate_gold_mapping, load_gold_structure

gold_path = resolved_gold_input or str(NCS_ROOT / 'data/gold/ncsMappings/gold_ncs_mapping_v1_TEMPLATE.csv')
gold_structure = load_gold_structure(gold_path)
evaluation_preview = evaluate_gold_mapping(gold_structure)
input_audit = evaluation_preview.to_dict()
assert input_audit['goldRows'] == 0
assert input_audit['precision'] is None and input_audit['finalCoverage'] is None
assert input_audit['gateStatus'] == 'NOT_EVALUATED'
input_audit

{'goldRows': 0,
 'adjudicatedRows': 0,
 'precision': None,
 'finalCoverage': None,
 'lowConfidenceRate': None,
 'gateStatus': 'NOT_EVALUATED',
 'reason': 'EMPTY_GOLD_INPUT',
 'goldValidatedFlag': False,
 'empiricalAnalysisAllowed': False,
 'promotionAllowed': False}

In [4]:
from p4_ncs.workflow.observed import run_stage

stage_manifest = run_stage('A4-05-EVALUATE', root=NCS_ROOT, duty_input_path=resolved_duty_input or None, gold_input_path=resolved_gold_input or None, schema_dir=resolved_schema_dir or None)
stage_manifest

{'manifestVersion': 'stage-manifest-v1',
 'runId': 'NCS_MAPPING_OBSERVED_20260806_01',
 'runMode': 'observed-dev',
 'stageId': 'A4-05-EVALUATE',
 'status': 'NOT_EVALUATED',
 'agentId': 'P4-A4-NCS',
 'branch': 'agent/p4-ncs-mapping-v2',
 'gitHead': 'afac9fd9e1854715759c7d893440cbe89e04c150',
 'contractVersion': '2.1.2',
 'schemaVersion': 'ncs-evaluation-v1',
 'dataVersion': 'observed-dev-20260806.1',
 'crawlReleaseId': 'CRAWL_20260806_03',
 'dataProvenance': 'OBSERVED_DEVELOPMENT_ONLY',
 'startedAt': '2026-08-06T08:49:41.104309Z',
 'completedAt': '2026-08-06T08:49:41.105765Z',
 'empiricalAnalysisAllowed': False,
 'promotionAllowed': False,
 'inputManifestSha256': 'c72ef16f5b80f38a1fb979e4b9720805034a8380538bddcd75b63331bbed0f2a',
 'parameterSha256': 'ee2a95b4cf722fbc15def7f7c955eba2248f681ec2a54ffef33e1869c3836e36',
 'rowCounts': {'goldInput': 0, 'adjudicatedGold': 0},
 'gateResults': [{'gateId': 'NCS_MAPPING_GOLD_READY',
   'status': 'NOT_EVALUATED',
   'evidencePath': 'ncs_mapping/dat

In [5]:
stage_root = NCS_ROOT / 'data/runs' / RUN_MODE / 'NCS_MAPPING_OBSERVED_20260806_01' / stage_manifest['stageId']
expected_artifacts = {'stage_manifest.json', 'stage_metrics.json', 'stage_quality.csv', 'CHECKSUMS.sha256'}
actual_artifacts = {path.name for path in stage_root.iterdir() if path.is_file()}
assert actual_artifacts == expected_artifacts
termination_summary = {'stageId': stage_manifest['stageId'], 'status': stage_manifest['status'], 'rowCounts': stage_manifest['rowCounts'], 'artifacts': sorted(actual_artifacts)}
termination_summary

{'stageId': 'A4-05-EVALUATE',
 'status': 'NOT_EVALUATED',
 'rowCounts': {'goldInput': 0, 'adjudicatedGold': 0},
 'artifacts': ['CHECKSUMS.sha256',
  'stage_manifest.json',
  'stage_metrics.json',
  'stage_quality.csv']}